# Día 3 · Evaluación final normal vs. HyDE

Integra la adjudicación humana, verifica la compuerta del juez v3 y reevalúa los 178 pares recuperados.

In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-03-experimento-hyde
!pip -q install pandas openpyxl openai scikit-learn

## 1. Subir tres archivos

Selecciona simultáneamente: `resultados_hyde_dia_03.zip`, `calibracion_juez_v3.zip` y `adjudicacion_relevancia_v3.xlsx`.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, pandas as pd, shutil
uploaded = files.upload()
original_zip = next(n for n in uploaded if n.startswith('resultados_hyde') and n.endswith('.zip'))
calibration_zip = next(n for n in uploaded if n.startswith('calibracion_juez_v3') and n.endswith('.zip'))
adjudication_xlsx = next(n for n in uploaded if n.startswith('adjudicacion_relevancia_v3') and n.endswith('.xlsx'))
input_dir=Path('/content/hyde_v1'); cal_dir=Path('/content/calibracion_juez_v3')
input_dir.mkdir(exist_ok=True); cal_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(original_zip) as z: z.extractall(input_dir)
with zipfile.ZipFile(calibration_zip) as z: z.extractall(cal_dir)
print('Archivos cargados correctamente')

## 2. Consolidar el Gold Standard y verificar la compuerta

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score
cal=pd.read_csv(cal_dir/'retrieval_results_rejudged_v3.csv')
adj=pd.read_excel(adjudication_xlsx,sheet_name='Adjudicacion',header=1)
keys=['question_id','chunk_id']
cal=cal.merge(adj[keys+['decision_final_relevancia','calidad_chunk']],on=keys,how='left',validate='one_to_one')
cal['gold_final']=cal.decision_final_relevancia.where(cal.decision_final_relevancia.notna(),cal.control_humano_relevante).astype(int)
y=cal.gold_final; p=cal.relevant.astype(int)
tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
metrics={'TP':int(tp),'TN':int(tn),'FP':int(fp),'FN':int(fn),'accuracy':accuracy_score(y,p),'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),'f1':f1_score(y,p,zero_division=0),'specificity':tn/(tn+fp),'kappa':cohen_kappa_score(y,p)}
display(pd.DataFrame([metrics]))
gate=metrics['accuracy']>=.70 and metrics['f1']>=.70
assert gate, 'El juez v3 no superó la compuerta.'
print('COMPUERTA APROBADA')

## 3. Cargar la clave temporal y reevaluar los 178 pares

In [ ]:
import os
from getpass import getpass
os.environ['OPENAI_API_KEY']=getpass('OPENAI_API_KEY: ')

In [ ]:
import subprocess
out=Path('/content/resultados_hyde_juez_v3'); out.mkdir(exist_ok=True)
shutil.copy(cal_dir/'judge_v3_checkpoint.jsonl',out/'judge_v3_checkpoint.jsonl')
subprocess.run(['python','-m','src.retrieval.rejudge_hyde_results','--results',str(input_dir/'retrieval_results_evaluated.csv'),'--questions','data/evaluation/gold_questions.csv','--output-dir',str(out)],check=True)
display(pd.read_csv(out/'metrics_by_method_v3.csv'))

## 4. Descargar resultados

In [ ]:
cal.to_csv(out/'gold_standard_audit_final.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([metrics]).to_csv(out/'judge_v3_validation_metrics.csv',index=False,encoding='utf-8-sig')
result_zip=shutil.make_archive('/content/resultados_finales_hyde_dia_03','zip',out)
files.download(result_zip)